**Lab 4**

Use word embeddings to improve prompts for Generative AI model. Retrieve similar words using word embeddings. Use the similar words to enrich a Gen AI prompt. Use the AI model to generate responses for the original and enriched prompts. Compare the outputs in terms of detail and relevance.

In [ ]:
!pip install -q gensim transformers torch nltk

In [ ]:
import nltk
import string
import gensim.downloader as ptwv

from transformers import pipeline
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download("punkt")

In [ ]:
print("Loading pre-trained word vectors...")

word_vectors = ptwv.load("glove-wiki-gigaword-100")

print("Word vectors loaded successfully")

In [ ]:
def replace_keyword_in_prompt(
    prompt,
    keyword,
    word_vectors,
    topn=1
):

    words = word_tokenize(prompt)

    enriched_words = []

    for word in words:

        cleaned_word = word.lower().strip(
            string.punctuation
        )

        if cleaned_word == keyword.lower():

            try:
                similar_words = word_vectors.most_similar(
                    cleaned_word,
                    topn=topn
                )

                if similar_words:
                    replacement_word = similar_words[0][0]

                    print(
                        f"Replacing '{word}' -> "
                        f"'{replacement_word}'"
                    )

                    enriched_words.append(
                        replacement_word
                    )

                    continue

            except KeyError:
                print(
                    f"'{keyword}' not found "
                    "in vocabulary."
                )

        enriched_words.append(word)

    enriched_prompt = " ".join(enriched_words)

    print(
        f"\nEnriched Prompt: "
        f"{enriched_prompt}"
    )

    return enriched_prompt

In [ ]:
print("Loading GPT-2 model...")

generator = pipeline(
    "text-generation",
    model="gpt2"
)

print("GPT-2 loaded successfully")

In [ ]:
def generate_response(
    prompt,
    max_length=100
):

    try:

        response = generator(
            prompt,
            max_length=max_length,
            num_return_sequences=1,
            truncation=True,
            pad_token_id=50256,
        )

        return response[0]["generated_text"]

    except Exception as e:

        print(
            f"Error generating response: {e}"
        )

        return None

In [ ]:
original_prompt = "Who is king?"

print(
    f"Original Prompt: "
    f"{original_prompt}"
)

In [ ]:
key_term = "king"

enriched_prompt = replace_keyword_in_prompt(
    original_prompt,
    key_term,
    word_vectors
)

In [ ]:
print(
    "\nGenerating response for "
    "the original prompt..."
)

original_response = generate_response(
    original_prompt
)

print(
    f"\nOriginal Prompt Response:\n"
    f"{original_response}"
)

In [ ]:
print(
    "\nGenerating response for "
    "the enriched prompt..."
)

enriched_response = generate_response(
    enriched_prompt
)

print(
    f"\nEnriched Prompt Response:\n"
    f"{enriched_response}"
)

In [ ]:
print("\nComparison of Responses:")

print(
    f"Original Prompt Response Length: "
    f"{len(original_response)}"
)

print(
    f"Enriched Prompt Response Length: "
    f"{len(enriched_response)}"
)

print(
    f"Original Prompt Response Detail: "
    f"{original_response.count('.')}"
)

print(
    f"Enriched Prompt Response Detail: "
    f"{enriched_response.count('.')}"
)

# Analysis

1. Word embeddings were used to identify semantically similar words.

2. The keyword "king" was replaced with a related word obtained from GloVe embeddings.

3. GPT-2 generated responses for both the original and enriched prompts.

4. The enriched prompt provides additional semantic context, which can influence the generated response.

5. Comparing response length and detail helps evaluate whether embedding-based prompt enrichment improves output quality.

6. Word embeddings can be used as a preprocessing step to create more informative prompts for Generative AI systems.